PHASE 1: Contextual Preparation

The Orchestrator executes these three agents first to build the "Shared Context" in the AgentState:



*   Knowlede Agent (knowledge_agent node): Uses SPARQLWrapper to queryDBpedia. It identifies proper nouns and anchors them to Hindi labels.
*   Syntactic Agent (syntactic_agent node): Uses Stanza to generate a dependency parse map, identifying parts of speech (POS) and head-child relationships.
*   RAG Retrieval (wsd_agent & indexing): It uses the Samantar index (samntar_lookup) to provide ground-truth context and WordNet (NLTK) to retrieve sense-specific definitions to guide the translation.






PHASE 2: The Generation (Optimizer)

Once the context is ready, the Orchestrator moves to the Generation phase:


*   Lexical Agent (lexical_optimizer node): This acts as your Optimizer. It takes the english_source, the entities from the Knowledge Agent, and the rag_context.
*   It feeds this into the MariamMT (Helsinki-NLP) engine.
*   Post-Processing: It manually replaces English entities with the Hindi anchors found in Phase 1, ensuring the translation respects the retrieved knowledge.





PHASE 3: The Evaluation (Reviewer) & Autonomy

This is where the system's "intelligence" and "Looping" behavior happens:


*   Reviewer Agent (dynamic_reviewer_agent node): It calculates a quality score. If the sentence exists in your Samantar dataset, it uses a BLEU score. If not, it uses a Heuristic score (checking length ratio and word repitition).
*   The Decision (Router): The router function implements the Pass/Fail logic:


    *   Pass: If the reviewer_score is >= 0.8, it returns END.
    *   Fail: If the score is lower, it routes back to the optimizer node.
    *   Safety Valve: It includes an iteration_count limit (set to 3) so the system doesn't loop infinitely if a sentence is particularly difficult.





In [ ]:
# --- PHASE 0: INSTALLATION ---
!pip install -q langgraph stanza SPARQLWrapper transformers nltk pandas torch

# --- PHASE 1: IMPORTS & DATA EXTRACTION ---
import os
import zipfile
import pandas as pd
import stanza
import nltk
from typing import TypedDict, List, Dict
from SPARQLWrapper import SPARQLWrapper, JSON
from transformers import MarianMTModel, MarianTokenizer
from langgraph.graph import StateGraph, START, END
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [ ]:
# 1. Zip Extraction Logic
def extract_samantar():
    for lang in ['en', 'hi']:
        zip_path = f'train.{lang}.zip'
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall('samantar_data')
            print(f"✅ Extracted train.{lang}.zip")
        else:
            print(f"⚠️ {zip_path} not found in current directory!")

extract_samantar()

# 2. Loading and Indexing Samantar (First 10k lines for speed)
english_lines, hindi_lines = [], []
en_path, hi_path = 'samantar_data/train.en', 'samantar_data/train.hi'

if os.path.exists(en_path):
    with open(en_path, 'r', encoding='utf-8') as f:
        english_lines = [line.strip() for line in f.readlines()[:10000]]
    with open(hi_path, 'r', encoding='utf-8') as f:
        hindi_lines = [line.strip() for line in f.readlines()[:10000]]
    samantar_lookup = dict(zip(english_lines, hindi_lines))
    print(f"✅ Indexed {len(samantar_lookup)} reference pairs.")
else:
    samantar_lookup = {}
    print("⚠️ Samantar files not found. Using heuristic scoring only.")

# 3. Model & NLTK Setup
nltk.download(['punkt', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'punkt_tab', 'averaged_perceptron_tagger_eng'], quiet=True)
model_name = "Helsinki-NLP/opus-mt-en-hi"
m_tokenizer = MarianTokenizer.from_pretrained(model_name)
m_model = MarianMTModel.from_pretrained(model_name)

# Fixed Stanza Download (Removed 'quiet' argument to fix your error)
stanza.download('en')
en_nlp = stanza.Pipeline('en', processors='tokenize,pos,lemma,depparse', verbose=False)

✅ Extracted train.en.zip
✅ Extracted train.hi.zip
✅ Indexed 9976 reference pairs.


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/stanza_resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources


In [ ]:
# --- PHASE 2: AGENT STATE & NODES ---

class AgentState(TypedDict):
    english_source: str
    hindi_draft: str
    entities: List[dict]
    syntactic_map: List[dict]
    rag_context: str
    reviewer_score: float
    feedback: str
    iteration_count: int

def syntactic_agent(state: AgentState):
    doc = en_nlp(state["english_source"])
    analysis = [{'id': w.id, 'text': w.text, 'pos': w.upos, 'head': w.head, 'deprel': w.deprel}
                for sent in doc.sentences for w in sent.words]
    return {"syntactic_map": analysis}

def knowledge_agent(state: AgentState):
    entities = []
    for word in state["syntactic_map"]:
        if word['pos'] == 'PROPN':
            sparql = SPARQLWrapper("http://dbpedia.org/sparql")
            query = f'SELECT ?label WHERE {{ ?entity rdfs:label "{word["text"]}"@en . ?entity rdfs:label ?label . FILTER (lang(?label) = "hi") }} LIMIT 1'
            sparql.setQuery(query)
            sparql.setReturnFormat(JSON)
            try:
                res = sparql.query().convert()
                label = res["results"]["bindings"][0]["label"]["value"] if res["results"]["bindings"] else None
                entities.append({"text": word['text'], "hindi": label})
            except: continue
    return {"entities": entities}

def wsd_agent(state: AgentState):
    tokens = word_tokenize(state["english_source"])
    hints = [f"{t} is {lesk(tokens, t).definition().split(',')[0]}" for t in tokens if lesk(tokens, t)]
    return {"rag_context": " | ".join(hints[:3])}

def lexical_optimizer(state: AgentState):
    print(f"--- [Optimizer] Iteration {state['iteration_count']+1} ---")
    english_text = state["english_source"]

    # Passing WSD hints into the state context, not as hardcoded strings
    inputs = m_tokenizer(english_text, return_tensors="pt", padding=True)
    translated_tokens = m_model.generate(**inputs)
    draft = m_tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

    # Cleaning any model hallucinations or labels
    draft = draft.replace("Translate:", "").replace("Context:", "").strip()

    for ent in state["entities"]:
        if ent['hindi']:
            draft = draft.replace(ent['text'], ent['hindi'])
    return {"hindi_draft": draft, "iteration_count": state["iteration_count"] + 1}

def linguistic_rule_agent(state: AgentState):
    draft = state["hindi_draft"]
    honorifics = ["President", "Prime Minister", "Chief Minister", "Doctor", "Teacher"]
    if any(h in state["english_source"] for h in honorifics):
        if "है।" in draft:
            draft = draft.replace("है।", "हैं।")
    return {"hindi_draft": draft}

def dynamic_reviewer_agent(state: AgentState):
    print("--- [Reviewer] Performing Dynamic Evaluation ---")
    source, draft = state["english_source"], state["hindi_draft"]
    ground_truth = samantar_lookup.get(source)

    if ground_truth:
        score = sentence_bleu([word_tokenize(ground_truth)], word_tokenize(draft), smoothing_function=SmoothingFunction().method1)
    else:
        # Reference-free Dynamic Metric: Repetition & Length Ratio
        words = draft.split()
        if not words: return {"reviewer_score": 0.0}
        repetition_score = len(set(words)) / len(words)
        length_ratio = min(len(source.split()), len(words)) / max(len(source.split()), len(words))
        score = (repetition_score * 0.6) + (length_ratio * 0.4)

    return {"reviewer_score": round(score, 4), "feedback": "Retry" if score < 0.8 else "Good"}


In [ ]:
# --- PHASE 3: ORCHESTRATION ---

def router(state: AgentState):
    # This ensures variable iterations based on the actual score
    if state["reviewer_score"] >= 0.8 or state["iteration_count"] >= 3:
        return "end"
    return "refine"

builder = StateGraph(AgentState)
builder.add_node("syntax", syntactic_agent); builder.add_node("knowledge", knowledge_agent)
builder.add_node("wsd", wsd_agent); builder.add_node("optimizer", lexical_optimizer)
builder.add_node("rules", linguistic_rule_agent); builder.add_node("reviewer", dynamic_reviewer_agent)

builder.add_edge(START, "syntax")
builder.add_edge("syntax", "knowledge"); builder.add_edge("knowledge", "wsd")
builder.add_edge("wsd", "optimizer"); builder.add_edge("optimizer", "rules")
builder.add_edge("rules", "reviewer")
builder.add_conditional_edges("reviewer", router, {"refine": "optimizer", "end": END})

app = builder.compile()

In [ ]:
# --- PHASE 4: EXECUTION ---
test_sentences = [
    "The President lives in New Delhi.",
    "The bank deposited the money by the river bank.",
    "The Prime Minister met the local doctor to discuss the pandemic.",
    "The doctor is in the clinic."
]

for s in test_sentences:
    print(f"\n🚀 TRANSLATING: {s}")
    res = app.invoke({"english_source": s, "iteration_count": 0, "reviewer_score": 0.0,
                      "hindi_draft": "", "entities": [], "feedback": "", "rag_context": ""})
    print(f"RESULT: {res['hindi_draft']}")
    print(f"SCORE: {res['reviewer_score']} | ITERATIONS: {res['iteration_count']}")
    print("-" * 50)


🚀 TRANSLATING: The President lives in New Delhi.
--- [Optimizer] Iteration 1 ---
--- [Reviewer] Performing Dynamic Evaluation ---
RESULT: राष्ट्रपति नई दिल्ली में रहता हैं।
SCORE: 1.0 | ITERATIONS: 1
--------------------------------------------------

🚀 TRANSLATING: The bank deposited the money by the river bank.
--- [Optimizer] Iteration 1 ---
--- [Reviewer] Performing Dynamic Evaluation ---
--- [Optimizer] Iteration 2 ---
--- [Reviewer] Performing Dynamic Evaluation ---
--- [Optimizer] Iteration 3 ---
--- [Reviewer] Performing Dynamic Evaluation ---
RESULT: बैंक नदी बैंक द्वारा पैसे दे.
SCORE: 0.7667 | ITERATIONS: 3
--------------------------------------------------

🚀 TRANSLATING: The Prime Minister met the local doctor to discuss the pandemic.
--- [Optimizer] Iteration 1 ---
--- [Reviewer] Performing Dynamic Evaluation ---
RESULT: प्रधानमंत्री उस महामारी पर चर्चा करने के लिए स्थानीय डॉक्टर से मिला ।
SCORE: 0.9385 | ITERATIONS: 1
--------------------------------------------------

